In [7]:
"""
OGI Fabric — CD PDE State Updater
==================================
Drop-in replacement for the GRU state update in FabricModule.receive_message().

Integrates Nelson Spence's navi-creative-determinant library.

Mathematical grounding:
  GRU (baseline): empirical convergence, Lemma 4.2 satisfied experimentally
  CD PDE (this):  MonotoneFixedPoint.lean — 15 theorems, zero sorry
                  Convergence to unique fixed point is VERIFIED, not asserted.

The compressed Goal-Gate message M(i→j) maps to the viability potential b.
The goal query q_j maps to the creative drive a.
The solution Φ of the CD PDE becomes the new module state.

Corten Research — Singleton, Dollinger & Spence
arXiv:2411.15832 Phase 2
"""

from __future__ import annotations

import numpy as np
import torch
import torch.nn as nn
from dataclasses import dataclass
from typing import Literal, List
import warnings

# ── CD library import ──────────────────────────────────────────────────────────
try:
    from cd import (
        solve_1d_picard,
        viability_threshold_1d,
        check_convergence,
        solution_type,
    )
    CD_AVAILABLE = True
except ImportError:
    CD_AVAILABLE = False
    warnings.warn(
        "navi-creative-determinant not found. FabricStateUpdater will use GRU only.",
        ImportWarning, stacklevel=2,
    )

# ── Dimensions — must match Fabric notebook ───────────────────────────────────
D_STATE = 64
D_MSG   = 64
D_QUERY = 32
CD_N    = D_STATE
CD_L    = 1.0
CD_C    = 10.0


@dataclass
class CDUpdateRecord:
    """One CD PDE state update record."""
    step:            int
    module_id:       str
    converged:       bool
    solution_type:   Literal['trivial', 'nontrivial']
    max_phi:         float
    beta_used:       float
    beta_star:       float
    above_threshold: bool
    iterations:      int

    @property
    def bifurcated(self) -> bool:
        return self.solution_type == 'nontrivial'


class CDStateUpdater:
    """
    CD PDE state update — drop-in replacement for GRUCell.

    call: new_state = updater.update(compressed, state, goal_query, step, module_id)
    """

    def __init__(
        self,
        d_state:    int   = D_STATE,
        L:          float = CD_L,
        c:          float = CD_C,
        beta_scale: float = 1.2,
    ):
        self.d_state    = d_state
        self.N          = d_state
        self.L          = L
        self.c          = c
        self.beta_scale = beta_scale
        self.records: List[CDUpdateRecord] = []

        if CD_AVAILABLE:
            self._beta_star_unit = viability_threshold_1d(self.L, b=1.0)

    def _message_to_viability(self, compressed: np.ndarray):
        msg = compressed.flatten()[:self.N]
        lo, hi = msg.min(), msg.max()
        b = np.ones(self.N) * 0.5 if hi - lo < 1e-8 else (msg - lo) / (hi - lo)
        return b, float(b.mean())

    def _query_to_drive(self, goal_query: np.ndarray) -> np.ndarray:
        q = goal_query.flatten()[:D_QUERY]
        a = np.interp(np.linspace(0, 1, self.N), np.linspace(0, 1, len(q)), q)
        return np.abs(a)

    def update(
        self,
        compressed:  torch.Tensor,
        state:       torch.Tensor,
        goal_query:  torch.Tensor,
        step:        int = 0,
        module_id:   str = '?',
    ) -> torch.Tensor:

        msg_np   = compressed.detach().numpy().flatten()
        query_np = goal_query.detach().numpy().flatten()

        b_field, b_mean = self._message_to_viability(msg_np)
        a_field         = self._query_to_drive(query_np)

        beta_star = viability_threshold_1d(self.L, b=b_mean) if b_mean > 1e-8 \
                    else float(self._beta_star_unit)
        beta_b    = self.beta_scale * beta_star

        try:
            x, Phi, info = solve_1d_picard(
                L=self.L, N=self.N,
                a=float(a_field.mean()),
                beta_b=beta_b, c=self.c,
            )
            converged = check_convergence(info)[0]
            sol_type  = solution_type(info)
            max_phi   = float(info.get('maxPhi', np.abs(Phi).max()))
            n_iter    = int(info.get('iterations', -1))

            # Strip Dirichlet boundary points: (N+2,) → (N,)
            if len(Phi) == self.N + 2:
                Phi = Phi[1:-1]

            # Normalize to same scale as incoming state
            state_norm = float(torch.norm(state).item())
            phi_norm   = float(np.linalg.norm(Phi))
            if phi_norm > 1e-8 and state_norm > 1e-8:
                Phi = Phi * (state_norm / phi_norm)

            new_state = torch.tensor(Phi, dtype=torch.float32).unsqueeze(0)

        except Exception as e:
            warnings.warn(f"CD solver failed step={step} module={module_id}: {e}. Using current state.")
            new_state = state
            converged = False
            sol_type  = 'trivial'
            max_phi   = 0.0
            beta_star = 0.0
            n_iter    = 0

        self.records.append(CDUpdateRecord(
            step=step, module_id=module_id, converged=converged,
            solution_type=sol_type, max_phi=max_phi,
            beta_used=beta_b, beta_star=beta_star,
            above_threshold=beta_b > beta_star, iterations=n_iter,
        ))
        return new_state

    def summary(self) -> dict:
        if not self.records:
            return {}
        n = len(self.records)
        return {
            'total_updates':    n,
            'converged':        sum(1 for r in self.records if r.converged),
            'convergence_rate': sum(1 for r in self.records if r.converged) / n,
            'nontrivial':       sum(1 for r in self.records if r.bifurcated),
            'bifurcation_rate': sum(1 for r in self.records if r.bifurcated) / n,
            'above_threshold':  sum(1 for r in self.records if r.above_threshold),
            'mean_max_phi':     np.mean([r.max_phi for r in self.records]),
        }


class FabricStateUpdater(nn.Module):
    """
    A/B state updater — wraps GRU and CDStateUpdater behind one interface.

    use_cd=False  →  GRU baseline (Lemma 4.2, empirical)
    use_cd=True   →  CD PDE      (MonotoneFixedPoint.lean, verified)

    This is the Phase 2 experimental design:
    same Fabric, same SAD probe, same goal-gating — two state updaters.

    Two line changes to FabricModule:

    In __init__():
        # OLD: self.gru = nn.GRUCell(d_msg, d_state)
        # NEW:
        self.state_updater = FabricStateUpdater(
            d_msg=d_msg, d_state=d_state, d_query=d_query, use_cd=False
        )

    In receive_message():
        # OLD: new_state = self.gru(compressed, state)
        # NEW:
        new_state = self.state_updater(
            compressed, state, q_j, step=step, module_id=self.module_id
        )
    """

    def __init__(
        self,
        d_msg:      int   = D_MSG,
        d_state:    int   = D_STATE,
        d_query:    int   = D_QUERY,
        use_cd:     bool  = False,
        beta_scale: float = 1.2,
    ):
        super().__init__()
        self.use_cd  = use_cd
        self.d_state = d_state

        # GRU — always available
        self.gru = nn.GRUCell(d_msg, d_state)

        # CD updater — requires Nelson's library
        if CD_AVAILABLE:
            self.cd = CDStateUpdater(d_state=d_state, beta_scale=beta_scale)
        else:
            self.cd = None
            if use_cd:
                warnings.warn("use_cd=True but CD library not installed. Falling back to GRU.")
                self.use_cd = False

    def forward(
        self,
        compressed:  torch.Tensor,
        state:       torch.Tensor,
        goal_query:  torch.Tensor,
        step:        int = 0,
        module_id:   str = '?',
    ) -> torch.Tensor:
        if self.use_cd and self.cd is not None:
            return self.cd.update(compressed, state, goal_query, step, module_id)
        else:
            return self.gru(compressed, state)

    def cd_summary(self) -> dict:
        return self.cd.summary() if self.cd is not None else {}

    def switch_to_cd(self):
        if not CD_AVAILABLE:
            raise RuntimeError("CD library not installed.")
        self.use_cd = True

    def switch_to_gru(self):
        self.use_cd = False


# ── Smoke test ─────────────────────────────────────────────────────────────────
if __name__ == '__main__':
    print(f"CD available: {CD_AVAILABLE}")
    print(f"D_STATE={D_STATE}, D_MSG={D_MSG}, D_QUERY={D_QUERY}")
    print()

    compressed = torch.randn(1, D_MSG)
    state      = torch.randn(1, D_STATE)
    query      = torch.randn(1, D_QUERY)

    # GRU
    u_gru = FabricStateUpdater(use_cd=False)
    s_gru = u_gru(compressed, state, query)
    print(f"GRU  shape={s_gru.shape}  norm={torch.norm(s_gru).item():.4f}")

    # CD
    if CD_AVAILABLE:
        u_cd = FabricStateUpdater(use_cd=True)
        s_cd = u_cd(compressed, state, query, step=0, module_id='A')
        print(f"CD   shape={s_cd.shape}  norm={torch.norm(s_cd).item():.4f}")
        print(f"CD summary: {u_cd.cd_summary()}")

CD available: True
D_STATE=64, D_MSG=64, D_QUERY=32

GRU  shape=torch.Size([1, 64])  norm=4.5834
CD   shape=torch.Size([1, 64])  norm=7.4060
CD summary: {'total_updates': 1, 'converged': 1, 'convergence_rate': 1.0, 'nontrivial': 1, 'bifurcation_rate': 1.0, 'above_threshold': 1, 'mean_max_phi': 2.009703417261311}


In [ ]:
# In SageMaker
import shutil
shutil.copy('/path/to/cd_state_updater.py', '/shared/ogi_navi_sad/cd_state_updater.py')

from cd_state_updater import FabricStateUpdater, CD_AVAILABLE
print(f"CD available: {CD_AVAILABLE}")

# Quick test
import torch
updater = FabricStateUpdater(use_cd=True)
compressed = torch.randn(1, 64)
state      = torch.randn(1, 64)
query      = torch.randn(1, 32)
new_state  = updater(compressed, state, query, step=0, module_id='A')
print(f"Shape: {new_state.shape}")
print(f"Norm:  {torch.norm(new_state).item():.4f}")
print(f"CD summary: {updater.cd_summary()}")